# Assignment 4: Autoencoder for Anomaly Detection (Credit Card Fraud)
**Aim:** Use an autoencoder to detect fraudulent transactions.

Train the autoencoder **only on normal transactions**. Fraud transactions are reconstructed badly
→ high reconstruction error → flagged as anomaly.

Dataset: 284807 transactions, features V1–V28 + Amount, `Class` (0 = normal, 1 = fraud).

## a. Import required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

I0000 00:00:1793640487.007160   34908 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640488.095685   34908 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## b. Upload / access the dataset

In [2]:
df = pd.read_csv("dataset/creditcardfraud-csv/creditcard.csv")
print(df.shape)
print(df["Class"].value_counts())
df.head()

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
X = df.drop(["Time", "Class"], axis=1).values
y = df["Class"].values.astype(int)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Train only on NORMAL transactions
x_train_normal = x_train[y_train == 0]
n_features = x_train.shape[1]
print("Features:", n_features, " Normal training samples:", x_train_normal.shape[0])

Features: 29  Normal training samples: 227451


## c. Encoder (29 → 64 → 32 → 16 latent)
## d. Decoder (16 → 32 → 64 → 29)

In [4]:
encoder = keras.Sequential([
    keras.Input(shape=(n_features,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),      # latent representation
], name="encoder")

decoder = keras.Sequential([
    keras.Input(shape=(16,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(n_features, activation="linear"),
], name="decoder")

autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.summary()

Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Sequential)            │ (None, 16)             │         4,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 29)             │         4,541 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,069 (35.43 KB)

 Trainable params: 9,069 (35.43 KB)

 Non-trainable params: 0 (0.00 B)

## e. Compile the model with Optimizer, Loss and Evaluation Metrics

In [5]:
autoencoder.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = autoencoder.fit(x_train_normal, x_train_normal,
                          epochs=10, batch_size=256,
                          validation_split=0.1, shuffle=True)

Epoch 1/10


  1/800 ━━━━━━━━━━━━━━━━━━━━ 7:55 595ms/step - loss: 0.8070 - mae: 0.6360

 32/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8964 - mae: 0.6387    

 65/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8762 - mae: 0.6240

100/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8053 - mae: 0.6046

136/800 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7581 - mae: 0.5868

... (363 outputs left out)

754/800 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0592 - mae: 0.1448

776/800 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0592 - mae: 0.1448

797/800 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0592 - mae: 0.1447

800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0591 - mae: 0.1447 - val_loss: 0.0554 - val_mae: 0.1405


In [6]:
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title("Autoencoder training loss"); plt.legend()
plt.show()

/tmp/ipykernel_34908/3738231248.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Anomaly detection using reconstruction error

In [7]:
test_recon = autoencoder.predict(x_test, verbose=0)
mse = np.mean(np.square(x_test - test_recon), axis=1)

# Threshold: 95th percentile of reconstruction error
threshold = np.percentile(mse, 95)
print("Threshold:", threshold)

y_pred = (mse > threshold).astype(int)   # 1 = fraud / anomaly
print("Anomalies detected:", y_pred.sum())

Threshold: 0.22840826874711315
Anomalies detected: 2849


In [8]:
plt.figure(figsize=(10, 4))
plt.scatter(np.where(y_test == 0)[0], mse[y_test == 0], s=2, label="Normal")
plt.scatter(np.where(y_test == 1)[0], mse[y_test == 1], s=8, color="red", label="Fraud")
plt.axhline(threshold, color="k", linestyle="--", label="Threshold")
plt.yscale("log"); plt.xlabel("Sample index"); plt.ylabel("Reconstruction error (log)")
plt.title("Anomaly Detection Results"); plt.legend(); plt.show()

/tmp/ipykernel_34908/3925532411.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Anomaly Detection Results"); plt.legend(); plt.show()


## Evaluate the model

In [9]:
print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))
print("ROC-AUC (using reconstruction error as score):", roc_auc_score(y_test, mse))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()

              precision    recall  f1-score   support

      Normal       1.00      0.95      0.97     56864
       Fraud       0.03      0.85      0.06        98

    accuracy                           0.95     56962
   macro avg       0.51      0.90      0.52     56962
weighted avg       1.00      0.95      0.97     56962

ROC-AUC (using reconstruction error as score): 0.9454652992316792


/tmp/claude-1000/-home-kalash-Projects-College/02ca1d49-a866-47a5-bc88-1ab49ed87890/scratchpad/venv/lib64/python3.14/site-packages/seaborn/matrix.py:309: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))


/tmp/ipykernel_34908/1579675913.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()
